In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from LegendreKANLayer import LegendreKANLayer
import random

batch_size = 64
# 设置设备为 GPU，如果可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 设置随机种子
seed = 5
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# 定义 Monge-Ampère 方程的源项和解析解（适用于三维）
def source_function(x, y, z):
    return (1 + x**2 + y**2 + z**2) * torch.exp(1.5 * (x**2 + y**2 + z**2))

def analytical_solution(x, y, z):
    return torch.exp((x**2 + y**2 + z**2) / 2)

# 定义求解模型
class LegenKAN(nn.Module):
    def __init__(self):
        super(LegenKAN, self).__init__()
        self.legenkan1 = LegendreKANLayer(3, 8, 6)
        self.legenkan2 = LegendreKANLayer(8, 8, 6)
        self.legenkan3 = LegendreKANLayer(8, 1, 6)

    def forward(self, x, y, z):
        xyz = torch.cat([x, y, z], dim=1)
        xyz = self.legenkan1(xyz)
        xyz = self.legenkan2(xyz)
        xyz = self.legenkan3(xyz)
        return xyz

# 初始化模型，并移动到设备
solver = LegenKAN().to(device)

# 初始采样点
x_values = torch.linspace(0, 1, 50, device=device).view(-1, 1)
y_values = torch.linspace(0, 1, 50, device=device).view(-1, 1)
z_values = torch.linspace(0, 1, 50, device=device).view(-1, 1)
X, Y, Z = torch.meshgrid(x_values.squeeze(), y_values.squeeze(), z_values.squeeze(), indexing='ij')
X, Y, Z = X.reshape(-1, 1), Y.reshape(-1, 1), Z.reshape(-1, 1)


# 内部点和边界点的掩码
boundary_mask = torch.isclose(X, torch.tensor(0.0, device=device)) | torch.isclose(X, torch.tensor(1.0, device=device)) | \
                torch.isclose(Y, torch.tensor(0.0, device=device)) | torch.isclose(Y, torch.tensor(1.0, device=device)) | \
                torch.isclose(Z, torch.tensor(0.0, device=device)) | torch.isclose(Z, torch.tensor(1.0, device=device))
interior_mask = ~boundary_mask

X_boundary, Y_boundary, Z_boundary = X[boundary_mask], Y[boundary_mask], Z[boundary_mask]
X_interior, Y_interior, Z_interior = X[interior_mask], Y[interior_mask], Z[interior_mask]

# 修正输入维度
X_boundary, Y_boundary, Z_boundary = X_boundary.view(-1, 1), Y_boundary.view(-1, 1), Z_boundary.view(-1, 1)
X_interior, Y_interior, Z_interior = X_interior.view(-1, 1), Y_interior.view(-1, 1), Z_interior.view(-1, 1)

# 启用梯度
X_interior.requires_grad = True
Y_interior.requires_grad = True
Z_interior.requires_grad = True

# 损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(solver.parameters(), lr=0.01)

# 自适应采样参数
num_adaptive_steps = 5
num_high_error_samples = 200
learning_rate_decay = 0.8
adaptive_sample_increment = 20

# 训练
epochs = 20000
alpha = 0.0001
previous_loss = float('inf')

for epoch in range(epochs):
    optimizer.zero_grad()

    # 计算边界和内部点的数值解
    numerical_boundary = solver(X_boundary, Y_boundary, Z_boundary)
    boundary_target = analytical_solution(X_boundary, Y_boundary, Z_boundary)

    # 边界损失，确保满足边界条件
    boundary_loss = criterion(numerical_boundary, boundary_target)

    numerical_interior = solver(X_interior, Y_interior, Z_interior)

    # 内部点的 Monge-Ampère 方程损失
    u_x = torch.autograd.grad(numerical_interior, X_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_y = torch.autograd.grad(numerical_interior, Y_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_z = torch.autograd.grad(numerical_interior, Z_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, X_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, Y_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_zz = torch.autograd.grad(u_z, Z_interior, grad_outputs=torch.ones_like(u_z), create_graph=True)[0]
    u_xy = torch.autograd.grad(u_x, Y_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xz = torch.autograd.grad(u_x, Z_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yz = torch.autograd.grad(u_y, Z_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]

    # 三维Hessian行列式
    hessian_det = u_xx * (u_yy * u_zz - u_yz**2) - u_xy * (u_xy * u_zz - u_xz * u_yz) + u_xz * (u_xy * u_yz - u_yy * u_xz)
    interior_loss = criterion(hessian_det, source_function(X_interior, Y_interior, Z_interior))

    # 总损失
    loss = boundary_loss + alpha * interior_loss

    # 反向传播与优化
    loss.backward(retain_graph=True)
    optimizer.step()

    # 检查损失是否增加，增加时减少学习率
    if loss.item() > previous_loss:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= learning_rate_decay
        print(f"Epoch [{epoch+1}/{epochs}], Loss increased. Reducing learning rate to: {param_group['lr']:.6f}")

    # 更新前一轮的总损失
    previous_loss = loss.item()
   
    # 每200个epoch输出损失信息
    if (epoch + 1) % 200 == 0:
        numerical_solution_global = solver(X, Y, Z)
        analytical_solution_global = analytical_solution(X, Y, Z)
        error_global = torch.abs(numerical_solution_global - analytical_solution_global)

        max_error = torch.max(error_global).item()
        mean_error = torch.mean(error_global).item()
        l2_error = torch.sqrt(torch.mean((numerical_solution_global - analytical_solution_global) ** 2)).item()

        print(f"Epoch [{epoch+1}/{epochs}], Total Loss: {loss.item():.4e}, "
            f"Boundary Loss: {boundary_loss.item():.4e}, Interior Loss: {interior_loss.item():.4e}, "
            f"Max Error: {max_error:.4e}, Average Error: {mean_error:.4e}, "
            f"L2 Error: {l2_error:.4e}")
        
    # 自适应采样
    if (epoch + 1) % (epochs // num_adaptive_steps) == 0:
        numerical_solution = solver(X_interior, Y_interior, Z_interior)
        analytical_solution_values = analytical_solution(X_interior, Y_interior, Z_interior)
        error = torch.abs(numerical_solution - analytical_solution_values)

        _, high_error_indices = torch.topk(error.view(-1), num_high_error_samples)
        X_high_error, Y_high_error, Z_high_error = X_interior[high_error_indices], Y_interior[high_error_indices], Z_interior[high_error_indices]

        delta = max(0.01, min(0.1, error.std().item()))
        X_new = torch.clamp(X_high_error + (torch.rand_like(X_high_error) - 0.5) * delta, 0, 1)
        Y_new = torch.clamp(Y_high_error + (torch.rand_like(Y_high_error) - 0.5) * delta, 0, 1)
        Z_new = torch.clamp(Z_high_error + (torch.rand_like(Z_high_error) - 0.5) * delta, 0, 1)

        X_interior = torch.cat([X_interior, X_new.requires_grad_(True)])
        Y_interior = torch.cat([Y_interior, Y_new.requires_grad_(True)])
        Z_interior = torch.cat([Z_interior, Z_new.requires_grad_(True)])
        num_high_error_samples += adaptive_sample_increment

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from LegendreKANLayer import LegendreKANLayer
import random


# 设置设备为 GPU，如果可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 设置随机种子
seed = 5
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)


def source_function(x, y, z, w):
    """
    f(x, y, z, w) = det(D^2 u) = (1 + x^2 + y^2 + z^2 + w^2) * exp(2 * (x^2 + y^2 + z^2 + w^2) / 2)
    """
    return (1 + x**2 + y**2 + z**2 + w**2) * torch.exp(2 * (x**2 + y**2 + z**2 + w**2) / 2)

def analytical_solution(x, y, z, w):
    """
    u(x, y, z, w) = exp((x^2 + y^2 + z^2 + w^2) / 2)
    """
    return torch.exp((x**2 + y**2 + z**2 + w**2) / 2)

# 定义求解模型
class LegenKAN(nn.Module):
    def __init__(self):
        super(LegenKAN, self).__init__()
        self.legenkan1 = LegendreKANLayer(4, 8, 6)
        self.legenkan2 = LegendreKANLayer(8, 8, 6)
        self.legenkan3 = LegendreKANLayer(8, 1, 6)

    def forward(self, x, y, z, w):
        xyzw = torch.cat([x, y, z, w], dim=1)
        xyzw = self.legenkan1(xyzw)
        xyzw = self.legenkan2(xyzw)
        xyzw = self.legenkan3(xyzw)
        return xyzw

# 初始化模型，并移动到设备
solver = LegenKAN().to(device)

# 初始采样点
x_values = torch.linspace(0, 1, 20, device=device).view(-1, 1)
y_values = torch.linspace(0, 1, 20, device=device).view(-1, 1)
z_values = torch.linspace(0, 1, 20, device=device).view(-1, 1)
w_values = torch.linspace(0, 1, 20, device=device).view(-1, 1)
X, Y, Z, W = torch.meshgrid(x_values.squeeze(), y_values.squeeze(), z_values.squeeze(), w_values.squeeze(), indexing='ij')
X, Y, Z, W = X.reshape(-1, 1), Y.reshape(-1, 1), Z.reshape(-1, 1), W.reshape(-1, 1)


# 内部点和边界点的掩码
boundary_mask = torch.isclose(X, torch.tensor(0.0, device=device)) | torch.isclose(X, torch.tensor(1.0, device=device)) | \
                torch.isclose(Y, torch.tensor(0.0, device=device)) | torch.isclose(Y, torch.tensor(1.0, device=device)) | \
                torch.isclose(Z, torch.tensor(0.0, device=device)) | torch.isclose(Z, torch.tensor(1.0, device=device)) | \
                torch.isclose(W, torch.tensor(0.0, device=device)) | torch.isclose(W, torch.tensor(1.0, device=device))
interior_mask = ~boundary_mask

X_boundary, Y_boundary, Z_boundary, W_boundary = X[boundary_mask], Y[boundary_mask], Z[boundary_mask], W[boundary_mask]
X_interior, Y_interior, Z_interior, W_interior = X[interior_mask], Y[interior_mask], Z[interior_mask], W[interior_mask]

# 修正输入维度
X_boundary, Y_boundary, Z_boundary, W_boundary = X_boundary.view(-1, 1), Y_boundary.view(-1, 1), Z_boundary.view(-1, 1), W_boundary.view(-1, 1)
X_interior, Y_interior, Z_interior, W_interior = X_interior.view(-1, 1), Y_interior.view(-1, 1), Z_interior.view(-1, 1), W_interior.view(-1, 1)

# 启用梯度
X_interior.requires_grad = True
Y_interior.requires_grad = True
Z_interior.requires_grad = True
W_interior.requires_grad = True


# 损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(solver.parameters(), lr=0.01)

# 自适应采样参数
num_adaptive_steps = 5
num_high_error_samples = 200
learning_rate_decay = 0.8
adaptive_sample_increment = 20

# 训练
epochs = 20000
alpha = 0.000000001
previous_loss = float('inf')

for epoch in range(epochs):
    optimizer.zero_grad()

    # 计算边界和内部点的数值解
    numerical_boundary = solver(X_boundary, Y_boundary, Z_boundary, W_boundary)
    boundary_target = analytical_solution(X_boundary, Y_boundary, Z_boundary, W_boundary)

    # 边界损失，确保满足边界条件
    boundary_loss = criterion(numerical_boundary, boundary_target)

    numerical_interior = solver(X_interior, Y_interior, Z_interior, W_interior)

    # 内部点的 Monge-Ampère 方程损失
    u_x = torch.autograd.grad(numerical_interior, X_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_y = torch.autograd.grad(numerical_interior, Y_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_z = torch.autograd.grad(numerical_interior, Z_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_w = torch.autograd.grad(numerical_interior, W_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]

    u_xx = torch.autograd.grad(u_x, X_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, Y_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_zz = torch.autograd.grad(u_z, Z_interior, grad_outputs=torch.ones_like(u_z), create_graph=True)[0]
    u_ww = torch.autograd.grad(u_w, W_interior, grad_outputs=torch.ones_like(u_w), create_graph=True)[0]

    u_xy = torch.autograd.grad(u_x, Y_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xz = torch.autograd.grad(u_x, Z_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xw = torch.autograd.grad(u_x, W_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    u_yz = torch.autograd.grad(u_y, Z_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_yw = torch.autograd.grad(u_y, W_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]

    u_zw = torch.autograd.grad(u_z, W_interior, grad_outputs=torch.ones_like(u_z), create_graph=True)[0]

    # 四维 Hessian 行列式
    hessian_det = (
        u_xx * (
            u_yy * (u_zz * u_ww - u_zw**2) -
            u_yz * (u_yz * u_ww - u_yw * u_zw) +
            u_yw * (u_yz * u_zw - u_yy * u_zw)
        )
        - u_xy * (
            u_xy * (u_zz * u_ww - u_zw**2) -
            u_xz * (u_yz * u_ww - u_yw * u_zw) +
            u_xw * (u_yz * u_zw - u_yy * u_zw)
        )
        + u_xz * (
            u_xy * (u_yw * u_zw - u_yz * u_ww) -
            u_xz * (u_xy * u_ww - u_xw * u_yw) +
            u_xw * (u_xy * u_zw - u_xz * u_yw)
        )
        - u_xw * (
            u_xy * (u_yz * u_zw - u_yw * u_zz) -
            u_xz * (u_xy * u_zw - u_xw * u_yz) +
            u_xw * (u_xy * u_zz - u_xz * u_yz)
        )
    )

    # 计算内部点的损失
    interior_loss = criterion(hessian_det, source_function(X_interior, Y_interior, Z_interior, W_interior))

    # 总损失
    loss = boundary_loss + alpha * interior_loss

    # 反向传播与优化
    loss.backward(retain_graph=True)
    optimizer.step()

    # 检查损失是否增加，增加时减少学习率
    if loss.item() > previous_loss:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= learning_rate_decay
        print(f"Epoch [{epoch+1}/{epochs}], Loss increased. Reducing learning rate to: {param_group['lr']:.6f}")

    # 更新前一轮的总损失
    previous_loss = loss.item()

    # 每200个epoch输出损失信息
    if (epoch + 1) % 200 == 0:
        numerical_solution_global = solver(X_interior, Y_interior, Z_interior, W_interior)
        analytical_solution_global = analytical_solution(X_interior, Y_interior, Z_interior, W_interior)
        error_global = torch.abs(numerical_solution_global - analytical_solution_global)

        max_error = torch.max(error_global).item()
        mean_error = torch.mean(error_global).item()
        l2_error = torch.sqrt(torch.mean((numerical_solution_global - analytical_solution_global) ** 2)).item()

        print(f"Epoch [{epoch+1}/{epochs}], Total Loss: {loss.item():.4e}, "
            f"Boundary Loss: {boundary_loss.item():.4e}, Interior Loss: {interior_loss.item():.4e}, "
            f"Max Error: {max_error:.4e}, Average Error: {mean_error:.4e}, "
            f"L2 Error: {l2_error:.4e}")
        
    # 自适应采样
    if (epoch + 1) % (epochs // num_adaptive_steps) == 0:
    # 计算数值解与解析解之间的误差
        numerical_solution = solver(X_interior, Y_interior, Z_interior, W_interior)
        analytical_solution_values = analytical_solution(X_interior, Y_interior, Z_interior, W_interior)
        error = torch.abs(numerical_solution - analytical_solution_values)
        num_high_error_samples = min(num_high_error_samples, error.numel())
        # 选择误差最高的点
        
        _, high_error_indices = torch.topk(error.view(-1), num_high_error_samples)
        X_high_error = X_interior[high_error_indices]
        Y_high_error = Y_interior[high_error_indices]
        Z_high_error = Z_interior[high_error_indices]
        W_high_error = W_interior[high_error_indices]

        # 计算扰动范围 delta
        delta = max(0.01, min(0.1, error.std().item()))
        X_new = torch.clamp(X_high_error + (torch.rand_like(X_high_error) - 0.5) * delta, 0, 1)
        Y_new = torch.clamp(Y_high_error + (torch.rand_like(Y_high_error) - 0.5) * delta, 0, 1)
        Z_new = torch.clamp(Z_high_error + (torch.rand_like(Z_high_error) - 0.5) * delta, 0, 1)
        W_new = torch.clamp(W_high_error + (torch.rand_like(W_high_error) - 0.5) * delta, 0, 1)

        # 将新采样点添加到内部点集合中
        X_interior = torch.cat([X_interior, X_new.requires_grad_(True)])
        Y_interior = torch.cat([Y_interior, Y_new.requires_grad_(True)])
        Z_interior = torch.cat([Z_interior, Z_new.requires_grad_(True)])
        W_interior = torch.cat([W_interior, W_new.requires_grad_(True)])

        # 增加高误差采样点数量
        num_high_error_samples += adaptive_sample_increment